## Guardrails in Agents

More on: https://openai.github.io/openai-agents-python/guardrails/

In [ ]:
# Import libraries
from dotenv import load_dotenv
from agents import Runner, trace, function_tool, Agent, OpenAIChatCompletionsModel, SQLiteSession
from openai import AsyncOpenAI

#### Load openai api key from environment

In [ ]:
# Load environment variables
load_dotenv(override=True)

#### Define LLM model

In [ ]:
agent_instructions = """You are a helpful assistant for Bioinformatics task
"""

bioinfo_agent = Agent(name="BioinformaticsAgent",
                    instructions=agent_instructions,
                    model="gpt-4.1-nano")


### Run agent without Guardrails

If you ask "hi, tell me about inception movie", the model would be able to respond. With Guardrails, we can prevent this. 

In [ ]:
# Define a session to maintain context across interactions
session = SQLiteSession("guardrails_agent_session")

# Can leverage chat interface for multiple turns
async def chat(message, history):
    # pass session object to maintain context
    result = await Runner.run(bioinfo_agent, message, session=session)
    return result.final_output

import gradio as gr
gr.ChatInterface(
    chat,
    title="Bioinformatics Assistant",
    description="Chat with the agent to get help with your bioinformatics tasks."
).launch()

#### Define Guardrail agent 

In [ ]:
from pydantic import BaseModel

# Define the output schema for the guardrail agent
class BioinformaticsOutput(BaseModel):
    not_a_bioinformatics_topic: bool
    reasoning: str

# Define the guardrail agent
guardrail_agent = Agent( 
    name="Guardrail check",
    instructions="Check if the user is asking you a bioinformatics related question",
    output_type=BioinformaticsOutput,
    model="gpt-4.1-nano"
)

### Define Guardrail function

In [ ]:
from agents import input_guardrail, GuardrailFunctionOutput

@input_guardrail(run_in_parallel=False)
async def guardrail_against_bioinformatics(ctx, agent, message):
    result = await Runner.run(guardrail_agent, message, context=ctx.context)
    print(result.final_output)
    not_a_bioinformatics_topic = result.final_output.not_a_bioinformatics_topic
    return GuardrailFunctionOutput(output_info={"found_bioinformatics_topic": result.final_output}
                                   ,tripwire_triggered=not_a_bioinformatics_topic)


### Define main agent

In [ ]:

agent_instructions = """You are a helpful assistant for Bioinformatics task
"""

bioinfo_agent = Agent(name="BioinformaticsAgent",
                    instructions=agent_instructions,
                    model="gpt-4.1-nano",
                    input_guardrails=[guardrail_against_bioinformatics]   
                    )



### Run in chat mode

In [ ]:
# Define a session to maintain context across interactions
session = SQLiteSession("guardrails_agent_session1")

# Can leverage chat interface for multiple turns
async def chat(message, history):
    try:
        result = await Runner.run(bioinfo_agent, message, session=session)
        return result.final_output
    except Exception as e:
        return f"Sorry, I can't assist with that. Reason: {str(e)}. Please ask Bioinformatics related questions."

   
import gradio as gr
gr.ChatInterface(
    chat,
    title="Bioinformatics Assistant",
    description="Chat with the agent to get help with your bioinformatics tasks."
).launch()